# Day 7：亲眼看抽屉分配、KV写入、回收与复用

先看 [四个角色与抽屉图解](../docs/DAY07_KV_LIFECYCLE.md)，个人记录见 [Day 7学习记录](../docs/DAY07_STUDY_LOG.md)。
Request是任务单，Scheduler是排班员，KV Manager管理抽屉目录，Attention Kernel执行注意力计算。
先运行数值标记的真实CPU页池，再运行随机小模型Engine；不安装或部署外部vLLM，不测GPU性能。
A/B/C是本实验请求，不沿用上一份调度Notebook的token预算。

In [ ]:
import os
import sys
from pathlib import Path

root = Path.cwd()
if root.name == 'learning':
    root = root.parent
if not (root / 'learning/day07_memory.py').is_file():
    raise RuntimeError('Open from repository root or learning directory.')
os.chdir(root)
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from learning.day07_memory import engine_lesson, small_config, storage_lesson

## 1. 先读抽屉账本，不先看公式

6个物理抽屉，每抽屉存2个位置，pool是384 bytes的真实CPU张量。
先预测：B归还块2/3后，free=[4,5,2,3]；C借3块，目录会是什么？
观察capacity是预留长度，lengths是各层已经写入的有效长度，二者不相等。
每个event都是依次执行后保存的快照，不是异步服务日志。

In [ ]:
storage_report = storage_lesson()
print('storage shape:', storage_report['storage_shape'])
print('pool bytes:', storage_report['storage_bytes'], '每抽屉:', storage_report['bytes_per_block'])
for event in storage_report['events']:
    print('\n', event['event'])
    print('抽屉:', ' '.join(f'[{i}:{owner}]' for i, owner in enumerate(event['owners'])))
    print('free队列:', event['free'])
    print('各请求目录/容量/有效长度:', event['requests'])
print('\nC位置4映射:', storage_report['C_position'])
print('C第0层按逻辑顺序读出的K:', storage_report['C_layer0_keys_in_logical_order'])
for key in ('released_handle_rejected', 'release_does_not_zero_page',
            'oom_without_mutation', 'all_blocks_free', 'storage_address_unchanged'):
    assert storage_report[key]
    print(key, storage_report[key])

## 2. 真正调用一次随机小模型，不只是减计数

A：prompt=[1,2,3]，max_tokens=3；B：prompt=[4,5,6]，max_tokens=1；C：prompt=[7,8]，max_tokens=2，arrival=1。
本引擎最大KV容量为P+G-1，A/B/C分别预留5/3/3个位置，最后一个输出不再次送回模型。
同一轮按(cache_position,input_length)分桶；同桶请求真正组成Batch，各自KV再拆回自己的页。
先预测：第一批A/B能否一起prefill？下一轮A decode和C prefill能否在此实现中直接合桶？
注意：Engine不调用simulate，也没有相同的token_budget配置。

In [ ]:
engine_report = engine_lesson()
print('实际模型输出ID（随机模型，不是语言质量）:', engine_report['outputs'])
print('预留最大KV位置:', engine_report['reserved_capacity'])
for batch in engine_report['batches']:
    print('一次真实model调用:', batch)
print('独立greedy对齐:', engine_report['independent_greedy_alignment'])
print('所有页已回收:', engine_report['all_blocks_free'])
print('物理pool地址不变:', engine_report['storage_address_unchanged'])

## 3. 故意在模型调用处抛异常，看finally是否归还抽屉

创建独立小引擎，在本次调用范围内替换forward使其抛错；不修改真实源码。
这验证现有Engine的同步异常清理，不是生产取消、异步CUDA或进程崩溃恢复。

In [ ]:
from unittest.mock import patch

import torch
from learning.mini_transformer import Model
from learning.mini_vllm.engine import Engine, Request

with torch.random.fork_rng(devices=[]):
    torch.manual_seed(123)
    failing_model = Model(small_config())
failing_engine = Engine(failing_model, blocks=4, block_size=2)
initial_address = failing_engine.pool.storage.data_ptr()
with patch.object(failing_model, 'forward', side_effect=RuntimeError('lesson failure')) as injected:
    try:
        failing_engine.run([Request('failure', [1, 2], 2)])
    except RuntimeError as error:
        assert str(error) == 'lesson failure'
        print('预期异常:', error)
    else:
        raise AssertionError('Injected model failure was not raised.')
    assert injected.call_count == 1
assert not failing_engine.pool.requests
assert len(failing_engine.pool.free) == 4
assert failing_engine.pool.storage.data_ptr() == initial_address
print('finally后live请求:', failing_engine.pool.requests)
print('free块数:', len(failing_engine.pool.free), '；pool本体仍保留。')

## 自测与答案

**Request是什么？** 保存输入、输出预算、已产生token等的请求记录，不是一份模型。

**Scheduler是什么？** 决定谁何时执行的策略；真实小Engine自己做准入与分桶，未复用离散simulate。

**KV Manager是什么？** 管理物理块、空闲队列、请求目录与有效长度；KV数值来自模型投影。

**Attention Kernel是什么？** 执行Q/K匹配和V汇集，不直接生成文字；还需FFN、LM Head、采样等环节。

**B释放后为什么pool内存没变小？** 归还的是所有权，池张量仍存在；本例也没有自动擦除数据。

**C会读到旧B内容吗？** 正确handle按有效长度读取，新C为空；写入后核对逻辑序列。原始storage访问不受此安全隔离保护。

**怎样优化？** 先核对所有权/数值/异常回收，再固定工作负载做profiling，逐项调整准入、预算、block大小或实际后端，并同时报告延迟与吞吐。

更多逐步debug位置、生命周期和官方阅读顺序见 [图解第5–9节](../docs/DAY07_KV_LIFECYCLE.md)。